# `needle` walkthrough

This notebook shows the public `needle` package and running Tasks without the CLI:

1. **`load_config`**: load and resolve a Hydra config
2. **`run`**: submit the DAG to the `b2luigi` backend (in-process)
3. **`load_snapshot`** / **`Estimator`**: load trained checkpoints and run aggregated inference
4. **`aggregate_siblings`**: call the aggregation logic directly, including a custom aggregator
5. **`train_single`**: train one estimator directly, bypassing the DAG fan-out
6. **`init`**: scaffold a new NEEDLE project
7. **`configure_law`** / **`configure_b2luigi`**: configure a backend from Python instead of `setup.sh`/`settings.json`

It uses `tests/conf_tests/config.yaml` (a tiny two-estimator DAG: `model_A`, and `model_B` which
`requires: ["model_A"]`) against the small parquet file bundled at
`examples/fair_universe_demo/test_data/`, so it runs standalone with no external data or
environment variables required. All run output is written to a temporary directory, so re-running
this notebook never leaves artifacts in the repo.


In [1]:
import tempfile
from pathlib import Path

import omegaconf
import torch

import needle

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "tests" / "conf_tests" / "config.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent  # running from examples/

CONFIG_PATH = REPO_ROOT / "tests" / "conf_tests" / "config.yaml"
DEMO_PARQUET = REPO_ROOT / "examples" / "fair_universe_demo" / "test_data" / "FAIR_Universe_HiggsML_data.parquet"
assert CONFIG_PATH.exists(), CONFIG_PATH
assert DEMO_PARQUET.exists(), DEMO_PARQUET

work_dir = Path(tempfile.mkdtemp(prefix="needle_api_demo_"))
print("Scratch results dir:", work_dir)


Scratch results dir: /tmp/needle_api_demo_8nl1g70q


## 1. `load_config`

Loads and fully resolves a Hydra config (defaults applied, `*_override` fields populated, DAG
validated) without needing a running `law`/`b2luigi` process. Returns the resolved
`needle.utils.config_schema.MainConfig` as an OmegaConf `DictConfig`.


In [2]:
cfg = needle.load_config(CONFIG_PATH)

print("Estimators:", list(cfg.estimators.keys()))
print("model_A requires:", cfg.estimators.model_A.requires)
print("model_B requires:", cfg.estimators.model_B.requires)
print("model_A systematics:", list(cfg.estimators.model_A.expands.systematics.keys()))
print("model_A folds/ensembles:", cfg.estimators.model_A.expands.folds, cfg.estimators.model_A.expands.ensembles)


Estimators: ['model_A', 'model_B']
model_A requires: None
model_B requires: ['model_A']
model_A systematics: ['nominal', 'up_qcd']
model_A folds/ensembles: {'num': 2, 'aggregation': {'method': 'mean', 'metric_key': None}} {'num': 2, 'aggregation': {'method': 'mean', 'metric_key': None}}


In [3]:
# Hydra override strings work the same way as `--hydra-overrides` on the CLI.
cfg_overridden = needle.load_config(CONFIG_PATH, overrides=["estimators.model_A.expands.folds=3"])
print("Default folds:", cfg.estimators.model_A.expands.folds)
print("Overridden folds:", cfg_overridden.estimators.model_A.expands.folds)


Default folds: {'num': 2, 'aggregation': {'method': 'mean', 'metric_key': None}}
Overridden folds: {'num': 3, 'aggregation': {'method': 'mean', 'metric_key': None}}


## 2. `run`

`run()` submits any task by class name to either backend. Here we point both estimators' dataset
paths at the bundled demo parquet, save the config, and run `MainTask` in-process via the
`b2luigi` backend (the default). This trains every estimator in the DAG (`model_A` and, because
it `requires: ["model_A"]`, `model_B`), then writes `config.yaml` and `dag_snapshot.json` into
`results_path`.


In [4]:
run_cfg = needle.load_config(CONFIG_PATH)
for name in run_cfg.estimators:
    run_cfg.estimators[name].dataset_override.paths = str(DEMO_PARQUET)
run_cfg._resolved = True

run_config_file = work_dir / "config.yaml"
omegaconf.OmegaConf.save(run_cfg, run_config_file, resolve=True)

result = needle.run(
    task="MainTask",
    backend="b2luigi",
    config_file=run_config_file,
    results_path=str(work_dir),
)
print(result)

INFO: NEEDLE-api.run (10:55:00) - Running with `b2luigi` workflow backend
INFO: Informed scheduler that task   b2luigi.MainTask__99914b932b   has status   PENDING
INFO: Informed scheduler that task   b2luigi.EstimatorTask_model_B_8a0f906e12   has status   PENDING
INFO: Informed scheduler that task   b2luigi.SystematicTask_model_B_nominal_43ba5637db   has status   PENDING
INFO: Informed scheduler that task   b2luigi.EnsembleTask_0_model_B_nominal_3c5493ea1d   has status   PENDING
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: Informed scheduler that task   b2luigi.FoldTask_0_model_B_0_fec67c247d   has status   PENDING
INFO: Informed scheduler that task   b2luigi.TrainingTask_0_model_B_0_fec67c247d   has status   PENDING
INFO: Informed scheduler that task   b2

/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=255` in the `DataLoader` to improve performance.
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:2

Epoch 0: 100%|██████████| 1/1 [00:00<00:00, 23.70it/s, v_num=d6b0]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 1/1 [00:00<00:00, 20.65it/s, v_num=d6b0]

...

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 1/1 [00:00<00:00, 25.39it/s, v_num=70eb]

2026/10/01 10:55:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/10/01 10:56:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
INFO: [pid 2297447] Worker Worker(salt=299947361, workers=1, host=portal2, username=kschmidt, pid=2297447) done      b2luigi.TrainingTask(estimator=model_B, systematic=nominal, ensemble=0, fold_index=0)
INFO:luigi-interface:[pid 2297447] Worker Worker(salt=299947361, workers=1, host=portal2, username=kschmidt, pid=2297447) done      b2luigi.TrainingTask(estimator=model_B, systematic=nominal, ensemble=0, fold_index=0)
INFO: Informed scheduler that task   b2luigi.TrainingTask_0_model_B_0_fec67c247d   has status   DONE
INFO:luigi-interface:Informed scheduler that task   b2luigi.TrainingTask_0_model_B_0_fec67c247d   has status   DONE
INFO: [pid 2297447] Worker Worker(salt=299947361, workers=1, host=portal2, username=kschmidt, pid=2297447) running   b2luigi.FoldTask(estimator=model_B, systematic=nominal, ensemble=0, 

RunResult(returncode=None)


In [5]:
cached_config = work_dir / "config.yaml"
snapshot_file = work_dir / "dag_snapshot.json"
print("Cached config written:", cached_config.exists())
print("DAG snapshot written:", snapshot_file.exists())
print(snapshot_file.read_text()[:400], "...")


Cached config written: True
DAG snapshot written: True
{
    "est=model_A&syst=up_qcd&ensem=0&fold=0": "/tmp/needle_api_demo_8nl1g70q/est__model_A/syst__up_qcd/ensem__0/fold__0/model.ckpt",
    "est=model_A&syst=up_qcd&ensem=0&fold=1": "/tmp/needle_api_demo_8nl1g70q/est__model_A/syst__up_qcd/ensem__0/fold__1/model.ckpt",
    "est=model_A&syst=up_qcd&ensem=1&fold=0": "/tmp/needle_api_demo_8nl1g70q/est__model_A/syst__up_qcd/ensem__1/fold__0/model.ckpt", ...


## 3. `load_snapshot` and `Estimator` — load checkpoints and run inference

`load_snapshot` unflattens `dag_snapshot.json` into `{systematic: {ensemble: {fold: ckpt_path}}}`
for one estimator. `Estimator` wraps that: it loads every checkpoint, then aggregates bottom-up
(folds → ensembles → systematics) using each level's `AggregationSpec` from the estimator's
config, producing a single `(mean, std)` prediction.


In [6]:
snapshot = needle.load_snapshot(work_dir, "model_A")
print("Systematics trained for model_A:", list(snapshot.keys()))
for systematic, ensembles in snapshot.items():
    for ensemble, folds in ensembles.items():
        print(f"  {systematic=} {ensemble=} folds={sorted(folds.keys())}")


Systematics trained for model_A: ['up_qcd']
  systematic='up_qcd' ensemble=0 folds=[0, 1]
  systematic='up_qcd' ensemble=1 folds=[0, 1]


In [7]:
model = needle.Estimator(work_dir, "model_A")

n_features = len(run_cfg.estimators.model_A.dataset_override.features_columns)
x = torch.rand(6, n_features)
mean, std = model(x)
print("mean.shape:", mean.shape, " std.shape:", std.shape)
mean, std


mean.shape: torch.Size([6, 1])  std.shape: torch.Size([6, 1])


(tensor([[-0.2045],
         [-0.1867],
         [-0.1923],
         [-0.2295],
         [-0.2120],
         [-0.2226]]),
 tensor([[0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.]]))

## 4. `aggregate_siblings`

`Estimator` calls `aggregate_siblings` internally, but it's also usable standalone
 
 1. Pass a list of sibling prediction tensors
 2. Add an aggregation method:
    - Builtins: `method="mean"/"sum"/"best"` (last one needs `metrics=`)
    - Custom: A dotted import path to a user-supplied callable implementing the `needle.api.Aggregator` protocol. A config-driven `AggregationSpec` (as `Estimator` uses internally) is passed the same way, read off directly: `aggregate_siblings(outputs, method=spec.method, metric_key=spec.metric_key)`.

`tests/api/test_eval.py::_weighted_mean` is a small worked example (also documented in
`docs/concepts/hydra_config.md`).

In [24]:
outputs = [torch.zeros(2, 1), torch.full((2, 1), 4.0)]
print("outputs", outputs)

mean_agg, std_agg = needle.aggregate_siblings(outputs, method="mean")
print("mean:", mean_agg.squeeze().tolist())

best_agg, _ = needle.aggregate_siblings(outputs, method="best", metrics=[0.9, 0.1])
print("best (lowest metric wins):", best_agg.squeeze().tolist())

# Dotted path to a user-supplied aggregator (weights: 3:1 in favor of the first sibling).
custom_agg, custom_std = needle.aggregate_siblings(
    outputs,
    method="tests.api.test_eval._weighted_mean",  # pass module by name
)
print("custom weighted mean:", custom_agg.squeeze().tolist())

outputs [tensor([[0.],
        [0.]]), tensor([[4.],
        [4.]])]
mean: [2.0, 2.0]
best (lowest metric wins): [4.0, 4.0]
custom weighted mean: [1.0, 1.0]


## 5. `train_single`

Bypasses `requires` and systematics/ensembles/folds fan-out entirely and trains exactly one
`TrainingTask(single=True)`. Useful for quick iteration on a single model. `b2luigi` only.

In [12]:
single_cfg = needle.load_config(CONFIG_PATH)
single_cfg.estimators.model_A.dataset_override.paths = str(DEMO_PARQUET)
single_cfg._resolved = True

single_config_file = work_dir / "single_config.yaml"
omegaconf.OmegaConf.save(single_cfg, single_config_file, resolve=True)

single_results = work_dir / "single"
needle.train_single(
    estimator="model_A",
    config_file=str(single_config_file),
    results_path=str(single_results),
)
print(sorted(p.name for p in single_results.rglob("*.ckpt")))


/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
INFO: NEEDLE-etl (11:00:26) - Loaded 1000 events with 1 column(s): ['PRI_lep_pt']
INFO: NEEDLE-etl (11:00:26) - Loaded 1000 events with 1 column(s): ['PRI_n_jets']
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/

/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=255` in the `DataLoader` to improve performance.
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.
/work/kschmidt/NEEDLE/needle-sbi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:2

Epoch 0: 100%|██████████| 1/1 [00:00<00:00, 31.24it/s, v_num=16fe]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 1/1 [00:00<00:00, 29.35it/s, v_num=16fe]

2026/10/01 11:00:26 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/10/01 11:00:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
INFO: NEEDLE-training (11:00:40) - Successfully completed training for Estimator 'model_A'


['epoch=0-step=1.ckpt', 'model.ckpt', 'model.ckpt']


## 6. `init` — scaffold a new NEEDLE project

`needle init` (and its Python equivalent `api.init`) writes `law.cfg`, `settings.json`,
`setup.sh` and a default `conf/` tree for a new project. Here it runs in the /tmp directory.


In [22]:
with tempfile.TemporaryDirectory(prefix="needle_init_demo_") as scaffold_dir:
    scaffold_result = needle.init(scaffold_dir, backend="both")

Created 'law.cfg' (LAW config file for managing Tasks)
Created 'index' (Index of needle.law_tasks, update with `law index`)
Created 'settings.json' (b2luigi settings file)
Created 'tasks.py' (b2luigi task index, required by the `b2luigi` CLI (incl. batch workers))
Created 'setup.sh' (Setup script for setting up the NEEDLE environment)
Created 'conf' (Config directory following the hydra schema)


## 7. `configure_law` / `configure_b2luigi`

Both set backend configuration from Python instead of sourcing `setup.sh` (`law`) or editing
`settings.json` (`b2luigi`). This is useful in a notebook or script where sourcing a shell file isn't an
option.

In [23]:
# law: sets LAW_HOME / LAW_CONFIG_FILE for this process.
needle.configure_law(law_home=str(REPO_ROOT / ".law"), law_config_file=str(REPO_ROOT / "law.cfg"))

# b2luigi: sets batch-system settings programmatically; `run(backend="b2luigi", ...)` calls this
# internally with `batch_system=`, so most callers never need it directly.
needle.configure_b2luigi(batch_system="local")